# End-to-End KTM + Off-Policy Pipeline

This notebook runs: processing -> KTM dataframe -> Gaussian behavior policy -> propensity tuples -> off-policy training -> visualizations.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'offpolicy_ktm_pipeline').exists():
    ROOT = ROOT.parent
if not (ROOT / 'offpolicy_ktm_pipeline').exists():
    raise RuntimeError('Run this notebook from inside the geometric_flow repo.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Repo root:', ROOT)

Repo root: /Users/samuelgirard/work/geometric_flow


In [2]:
from offpolicy_ktm_pipeline.src.dataframe_processing import process_pix_dataset, build_ktm_dataframe
from offpolicy_ktm_pipeline.src.gaussian_regression import fit_gaussian_by_order
from offpolicy_ktm_pipeline.src.propensity import build_offpolicy_dataset
from offpolicy_ktm_pipeline.src.estimators_and_training import (
    train_global_policy,
    get_policy_coefficients,
    mc_policy_value,
    optimal_irt_value,
)
from offpolicy_ktm_pipeline.src.visualization import (
    plot_training_history,
    build_policy_curves,
    plot_policy_curves,
)

In [3]:
pix_input = ROOT / 'data' / 'pix_data.csv'
if not pix_input.exists():
    raise FileNotFoundError(f'Missing input CSV: {pix_input}')

stats = process_pix_dataset(
    input_csv=str(pix_input),
    output_dir=str(ROOT / 'pix_mapping'),
)
stats

{'rows': 3222679,
 'num_users': 100000,
 'num_challenges': 1657,
 'num_skills': 499,
 'unknown_answer_result_rows': 35554,
 'processed_csv': '/Users/samuelgirard/work/geometric_flow/pix_mapping/pix_processed.csv',
 'irt_csv': '/Users/samuelgirard/work/geometric_flow/pix_mapping/pix_irt.csv',
 'irt_rows': 3187125,
 'irt_outcome_csv': '/Users/samuelgirard/work/geometric_flow/pix_mapping/pix_irt_outcome.csv',
 'irt_outcome_rows': 3187125}

In [4]:
df_ktm = build_ktm_dataframe(
    processed_csv=str(ROOT / 'pix_mapping' / 'pix_processed.csv'),
    sample_users=10000,  # set 0 to use all users
    seed=42,
    fit_intercept=False,
)
df_ktm.head(), df_ktm.shape

(         user  answer_number  item  skill  correct  proficiency  difficulties  \
 3202190     0              1     0      1        1    -1.110249     -4.045991   
 3202191     0              2     1     53        1    -1.110249     -3.936940   
 3202192     0              3     6      2        1    -1.110249     -2.752612   
 3202193     0              4     4      3        1    -1.110249     -3.114941   
 3202194     0              5    68     65        0    -1.110249     -1.031771   
 
          sequence_position  sequence_length  order_sequence  
 3202190                  0               32               0  
 3202191                  1               32               1  
 3202192                  2               32               2  
 3202193                  3               32               3  
 3202194                  4               32               4  ,
 (318717, 10))

In [5]:
fit_df = fit_gaussian_by_order(
    df_ktm,
    mu_degree=1,
    sigma_degree=2,
    min_obs_per_order=200,
)
fit_path = ROOT / 'pix_mapping' / 'ktm_gaussian_fit_structured.csv'
fit_df.to_csv(fit_path, index=False)
fit_df.head(), fit_df.shape, fit_path

(    subset  order_sequence  order_min  order_max  n_obs  mu_degree  \
 0  order_0               0          0          0  10000          1   
 1  order_1               1          1          1  10000          1   
 2  order_2               2          2          2   9999          1   
 3  order_3               3          3          3   9999          1   
 4  order_4               4          4          4   9998          1   
 
    sigma_degree       nll      rmse       mae  success  \
 0             2  1.042140  0.691754  0.515114     True   
 1             2  1.157408  0.770495  0.693859     True   
 2             2  0.774371  0.525449  0.348316     True   
 3             2  0.824297  0.552566  0.451163     True   
 4             2  0.980139  0.645451  0.568932     True   
 
                                             message  beta_mu_0  beta_mu_1  \
 0  CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL  -3.579445   0.180966   
 1  CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL  -3.251

In [6]:
offpolicy_df = build_offpolicy_dataset(df_ktm, fit_df, sigma_floor=1e-4)
offpolicy_path = ROOT / 'pix_mapping' / 'ktm_offpolicy_structured.csv'
offpolicy_df.to_csv(offpolicy_path, index=False)
offpolicy_df.head(), offpolicy_df.shape, offpolicy_path

(   sequence  user  item  correct    reward  answer_number  sequence_position  \
 0         0     0     0        1  0.000000              1                  0   
 1         0     0     1        1  0.109051              2                  1   
 2         0     0     6        1  1.293380              3                  2   
 3         0     0     4        1  0.931051              4                  3   
 4         0     0    68        0  0.000000              5                  4   
 
    order_sequence  sequence_length  proficiency  difficulties   subset  \
 0               0               32    -1.110249     -4.045991  order_0   
 1               1               32    -1.110249     -3.936940  order_1   
 2               2               32    -1.110249     -2.752612  order_2   
 3               3               32    -1.110249     -3.114941  order_3   
 4               4               32    -1.110249     -1.031771  order_4   
 
      mu_hat  sigma_hat  log_propensity  propensity  
 0 -3.

In [7]:
policy, history_df, coeffs = train_global_policy(
    offpolicy_df,
    fit_df=fit_df,
    objective='snips',
    denominator='mixture',
    mu_degree=1,
    sigma_degree=2,
    epochs=2,   # increase for stronger training
    lr=0.005,
    batch_size=8192,
    num_workers=0,
)
history_df.tail()

,epoch,train_loss,objective,denominator,ips,snips,cips,csnips,ess,ess_clip,mean_w,std_w
0,0,NaN,snips,mixture,0.481951,0.458381,0.457108,0.595296,25908.777344,31727.898438,1.051420,3.534639
1,1,-0.567818,snips,mixture,0.688117,0.684796,0.679182,0.764875,35876.257812,40221.199219,1.004849,2.821428
2,2,-0.792596,snips,mixture,0.876055,0.902588,0.874988,0.922450,47283.574219,48775.289062,0.970603,2.325514


In [8]:
beta_mu, beta_sigma = get_policy_coefficients(policy)
theta = offpolicy_df['proficiency'].to_numpy(dtype=float)
delta_min = float(offpolicy_df['difficulties'].min())
delta_max = float(offpolicy_df['difficulties'].max())

mc_val = mc_policy_value(
    theta=theta,
    beta_mu=beta_mu,
    beta_sigma=beta_sigma,
    delta_min=delta_min,
    n_mc=600,
    seed=42,
)
opt_val = optimal_irt_value(theta=theta, delta_min=delta_min, delta_max=delta_max)
print({'mc_policy_value': mc_val, 'irt_optimal_value': opt_val})

{'mc_policy_value': 0.7222661785767917, 'irt_optimal_value': 2.242317418106575}


In [ ]:
_ = plot_training_history(history_df, title='Training Metrics (SNIPS, mixture denominator)')
plt.show()

last_order = int(fit_df['order_sequence'].max())
brow = fit_df[fit_df['order_sequence'] == last_order].iloc[0]
b_mu = np.array([brow['beta_mu_0'], brow['beta_mu_1']], dtype=float)
b_sigma = np.array([brow['beta_sigma_0'], brow['beta_sigma_1'], brow['beta_sigma_2']], dtype=float)

theta_grid = np.linspace(float(offpolicy_df['proficiency'].min()), float(offpolicy_df['proficiency'].max()), 400)
curves = build_policy_curves(
    theta_grid=theta_grid,
    beta_mu_behavior=b_mu,
    beta_sigma_behavior=b_sigma,
    beta_mu_learned=beta_mu,
    beta_sigma_learned=beta_sigma,
)
d_grid = np.linspace(delta_min, delta_max, 1200)
z = theta_grid[:, None] - d_grid[None, :]
p = 1.0 / (1.0 + np.exp(-z))
exp_reward = p * (d_grid[None, :] - delta_min)
opt_delta_curve = d_grid[np.argmax(exp_reward, axis=1)]

_ = plot_policy_curves(
    theta_grid=theta_grid,
    behavior_mu=curves['behavior_mu'],
    behavior_sigma=curves['behavior_sigma'],
    learned_mu=curves['learned_mu'],
    learned_sigma=curves['learned_sigma'],
    optimal_delta=opt_delta_curve,
    y_min=-4,
    y_max=4,
    title='Behavior vs Learned Policy (with IRT optimal curve)',
)
plt.show()